# Weather Analysis Project

Tracker for weather data analysis for agricultural fields.

## Objectives

- Download NASA POWER weather data for field locations
- Calculate Growing Degree Days (GDD)
- Analyze temperature, precipitation, and solar radiation patterns
- Support crop modeling and irrigation planning

## Data Sources

- NASA POWER (POWER Data Access Viewer) - https://power.larc.nasa.gov/
- Field boundaries: synthetic Michigan corn fields (USDA NASS Crop Sequence Boundaries style)

## Progress

In [ ]:
import pandas as pd
import geopandas as gpd

# Load field boundaries
fields = gpd.read_file('data/michigan_fields.geojson')
print("Michigan Fields:")
print(fields[['field_id', 'area_acres', 'county']].to_string(index=False))

In [ ]:
# Load weather data with datetime parsing
weather = pd.read_csv('data/michigan_weather_2024.csv', parse_dates=['date'])
print(f"\nWeather Data:")
print(f"  Fields: {weather['field_id'].nunique()}")
print(f"  Date range: {weather['date'].min().date()} to {weather['date'].max().date()}")
print(f"  Total records: {len(weather)}")
print(f"  Date column dtype: {weather['date'].dtype}")
print(f"\nColumns: {weather.columns.tolist()}")

In [ ]:
# Filter to growing season (already done by API query, but verify)
growing_season = weather[(weather['date'] >= '2024-05-01') & (weather['date'] <= '2024-09-30')]
print(f"Growing season records: {len(growing_season)}")
print(f"Verified: Date column is datetime64[ns]")

## Findings

### Field Summary
- 3 Michigan corn fields downloaded
- Field sizes range from ~1,076 to ~2,669 acres
- Located in Shiawassee, Gratiot, and Saginaw counties

### Weather Data Summary
- Date column properly formatted as datetime64[ns]
- Covers 2024 growing season (May 1 - September 30)
- 153 days per field × 3 fields = 459 total records
- Parameters: T2M (mean temp), T2M_MAX, T2M_MIN, PRECTOTCORR (precipitation), ALLSKY_SFC_SW_DWN (solar radiation), RH2M (humidity)

## Next Steps

- Calculate Growing Degree Days (GDD) for corn
- Analyze temperature extremes and precipitation patterns
- Visualize weather trends across fields
- Compare with historical norms

## Time Series Visualization

In [ ]:
from IPython.display import Image, display

display(Image(filename='data/michigan_weather_timeseries.png'))

### Visualization Notes
- **Top panel**: Daily temperature range (T2M_MAX/T2M_MIN) with 7-day rolling average
- **Bottom panel**: Daily precipitation with 7-day rolling average
- Gray dotted lines at 10°C and 30°C show corn GDD base/cap temperatures
- 7-day rolling average smooths daily variation to show seasonal trends

## Historical Comparison (5-Year Baseline)

In [ ]:
# Load historical data (2019-2023)
historical = pd.read_csv('data/michigan_weather_historical.csv', parse_dates=['date'])
print(f"Historical data: {len(historical)} records")
print(f"Date range: {historical['date'].min().date()} to {historical['date'].max().date()}")
print(f"Years: {historical['date'].dt.year.unique()}")

## Anomaly Detection Results

In [ ]:
display(Image(filename='data/michigan_weather_anomaly.png'))

### Key Findings

#### Prolonged Dry Spells Detected:
| Field | Duration | Dates |
|-------|----------|-------|
| MI2600001 (Shiawassee) | 10 days | Sep 8-17, 2024 |
| MI2600002 (Gratiot) | 12 days | Sep 8-19, 2024 |
| MI2600003 (Saginaw) | 16 days | Sep 6-21, 2024 |

All three fields experienced a significant late-season dry spell in September 2024, with MI2600003 (Saginaw County) experiencing the longest at 16 consecutive days without rain.

#### No Other Significant Anomalies Detected:
- No late-season frost events
- No extreme heat stress events (>32°C for 5+ consecutive days)
- Rainfall deficits not statistically significant at monthly level

## Weather Dashboard Plots

In [ ]:
display(Image(filename='output/dashboard_assets/weather_trends.png'))

### Dashboard Summary

| Field | County | Cumulative GDD | Cumulative Precip |
|-------|--------|----------------|------------------|
| MI2600001 | Shiawassee | 1,474 | 394mm |
| MI2600002 | Gratiot | 1,179 | 466mm |
| MI2600003 | Saginaw | 1,221 | 371mm |

#### Key Observations:
- All fields reached **tasseling** stage (~1,150 GDD) by late July
- MI2600001 accumulated the most GDD (1,474), likely due to its southern location
- MI2600002 received the most precipitation (466mm), MI2600003 the least (371mm)
- Late September dry spell may have impacted late-season grain fill in MI2600003

## Next Steps

- Assess impact of September dry spell on late-season crop development
- Compare yield implications with historical data
- Add soil moisture data for irrigation planning